# Germline Analysis Blueprint 

This blueprint shows how to run a standard Germline workflow on whole exome data from `fastq` to `vcf`. We will start by downloading the sample reads and a set of BWA-indexed reference files. Then we will align the reads to the reference using BWA-MEM and call the variant using DeepVariant. The diagram below outlines these steps. 

![fq2bam_diagram](images/pbworkflow.png)


To process the exome samples, we will use a GPU accelerated software suite called [Parabricks](https://docs.nvidia.com/clara/parabricks/latest/index.html). It contains over 25 popular secondary analysis tools for manipulating DNA and RNA data, including tools for alignment (BWA, Giraffe, Minimap2, STAR, etc.), variant calling (DeepVariant, HaplotypeCaller, DeepSomatic, StarFusion, etc.), and post processing steps such as quality checks and gvcf processing. Each tool offers nearly identical output to the CPU versions, but with speedups of up to 100x. For the full list of tools, see the [documentation](https://docs.nvidia.com/clara/parabricks/latest/toolreference.html). 

In this blueprint, we are running inside the Parabricks 4.4.0 Docker container which can be found in the [NVIDIA GPU Container (NGC) Registry](https://catalog.ngc.nvidia.com/orgs/nvidia/teams/clara/containers/clara-parabricks). 

## Download the dataset

The data set used in this lab is the whole exome sample NA12878 from the [NIH](https://ftp-trace.ncbi.nlm.nih.gov/ReferenceSamples/giab/data_indexes/NA12878/sequence.index.NA12878_Illumina_HiSeq_Exome_Garvan_fastq_09252015) sequenced on Illumina. The fastq files for this sample, as well as the HG38 reference files (already indexed) can be downloaded using the `download_data.sh` script. 

In [1]:
%%sh 

./scripts/download_data.sh 

--2026-03-02 10:45:02--  ftp://ftp-trace.ncbi.nih.gov/ReferenceSamples/giab/data/NA12878/Garvan_NA12878_HG001_HiSeq_Exome/NIST7035_TAAGGCGA_L001_R1_001.fastq.gz
           => ‘NIST7035_TAAGGCGA_L001_R1_001.fastq.gz’
Resolving ftp-trace.ncbi.nih.gov (ftp-trace.ncbi.nih.gov)... 130.14.250.12, 130.14.250.31, 130.14.250.10, ...
Connecting to ftp-trace.ncbi.nih.gov (ftp-trace.ncbi.nih.gov)|130.14.250.12|:21... connected.
Logging in as anonymous ... Logged in!
==> SYST ... done.    ==> PWD ... done.
==> TYPE I ... done.  ==> CWD (1) /ReferenceSamples/giab/data/NA12878/Garvan_NA12878_HG001_HiSeq_Exome ... done.
==> SIZE NIST7035_TAAGGCGA_L001_R1_001.fastq.gz ... 1914722761
==> PASV ... done.    ==> RETR NIST7035_TAAGGCGA_L001_R1_001.fastq.gz ... done.
Length: 1914722761 (1.8G) (unauthoritative)

NIST7035_TAAGGCGA_L 100%[===================>]   1.78G  27.4MB/s    in 71s     

2026-03-02 10:46:14 (25.6 MB/s) - ‘NIST7035_TAAGGCGA_L001_R1_001.fastq.gz’ saved [1914722761]

--2026-03-02 10:46:14-- 

It can take up to 15 minutes to download and organize the data into the correct structure for this blueprint. In the meantime, let's discuss the data being downloaded. 

When the script completes, the `data` directory should contain two `.fastq.gz` files as a `ref` folder containing all the reference files, and an empty `output` folder where the output files from this workflow will be stored.  

```
data
├── NIST7035_TAAGGCGA_L001_R1_001.fastq.gz
├── NIST7035_TAAGGCGA_L001_R2_001.fastq.gz
└── ref
    ├── Homo_sapiens_assembly38.dict
    ├── Homo_sapiens_assembly38.fasta
    ├── Homo_sapiens_assembly38.fasta.amb
    ├── Homo_sapiens_assembly38.fasta.ann
    ├── Homo_sapiens_assembly38.fasta.bwt
    ├── Homo_sapiens_assembly38.fasta.fai
    ├── Homo_sapiens_assembly38.fasta.pac
    ├── Homo_sapiens_assembly38.fasta.sa
    ├── Homo_sapiens_assembly38.known_indels.vcf.gz
    └── Homo_sapiens_assembly38.known_indels.vcf.gz.tbi
```

Let's verify that the data downloaded correctly by running `ls` on the `data` directory. 

In [2]:
%%sh 

ls data

NIST7035_TAAGGCGA_L001_R1_001.fastq.gz
NIST7035_TAAGGCGA_L001_R2_001.fastq.gz
output
ref


Let's verify that the reference files downloaded correctly by running `ls` on the `ref` folder. 

In [3]:
%%sh 

ls data/ref

Homo_sapiens_assembly38.dict
Homo_sapiens_assembly38.fasta
Homo_sapiens_assembly38.fasta.amb
Homo_sapiens_assembly38.fasta.ann
Homo_sapiens_assembly38.fasta.bwt
Homo_sapiens_assembly38.fasta.fai
Homo_sapiens_assembly38.fasta.pac
Homo_sapiens_assembly38.fasta.sa
Homo_sapiens_assembly38.known_indels.vcf.gz
Homo_sapiens_assembly38.known_indels.vcf.gz.tbi


Now the data is downloaded and organized for downstream analysis. 

## Align the fastq files

In this section, we will use the [Parabricks fq2bam](https://docs.nvidia.com/clara/parabricks/latest/documentation/tooldocs/man_fq2bam.html) tool to align our `.fastq` files to the reference. 

Parabricks fq2bam is a wrapper for [BWA-MEM](https://github.com/lh3/bwa) that is optimized to run on the GPU, providing up to 60x speedups compared to the CPU-only version. It will also sort the output and can be configured to mark duplicates and recalibrate base quality scores. Below is a diagram showing how the steps within fq2bam are connected. 

![fq2bam_diagram](images/fq2bam.png)

In [5]:
%%sh 

# Make an output directory if it doesn't already exist
mkdir -p data/output

docker run --rm --gpus all \
    --volume $(pwd)/data:/workdir \
    --workdir /workdir \
    nvcr.io/nvidia/clara/clara-parabricks:4.6.0-1 \
    pbrun fq2bam \
    --ref ref/Homo_sapiens_assembly38.fasta \
    --in-fq NIST7035_TAAGGCGA_L001_R1_001.fastq.gz NIST7035_TAAGGCGA_L001_R2_001.fastq.gz \
    --knownSites ref/Homo_sapiens_assembly38.known_indels.vcf.gz \
    --out-bam output/NIST7035_TAAGGCGA_L001_R1_001.bam \
    --out-recal-file output/recal.txt \
    --gpusort --gpuwrite 


[Parabricks Options Mesg]: Checking argument compatibility
[Parabricks Options Mesg]: Auto mode: Setting --bwa-nstreams and device memory parameters automatically based on
available GPU memory. These settings are optimized for the GRCh38 reference genome and may need manual adjustment for
other references or optimal performance. For manual configuration guidance, see the fq2bam documentation.
[Parabricks Options Mesg]: Set --bwa-options="-K #" to produce compatible pair-ended results with previous versions of
fq2bam or BWA MEM.
[Parabricks Options Mesg]: Automatically generating ID prefix
[Parabricks Options Mesg]: Read group created for /workdir/NIST7035_TAAGGCGA_L001_R1_001.fastq.gz and
/workdir/NIST7035_TAAGGCGA_L001_R2_001.fastq.gz
[Parabricks Options Mesg]: @RG\tID:H7AP8ADXX.1\tLB:lib1\tPL:bar\tSM:sample\tPU:H7AP8ADXX.1


[PB Info 2026-Mar-02 18:55:06] ------------------------------------------------------------------------------
[PB Info 2026-Mar-02 18:55:06] ||                 Parabricks accelerated Genomics Pipeline                 ||
[PB Info 2026-Mar-02 18:55:06] ||                              Version 4.6.0-1                             ||
[PB Info 2026-Mar-02 18:55:06] ||                      GPU-PBBWA mem, Sorting Phase-I                      ||
[PB Info 2026-Mar-02 18:55:06] ------------------------------------------------------------------------------
[PB Info 2026-Mar-02 18:55:06] Mode = pair-ended-gpu
[PB Info 2026-Mar-02 18:55:06] Running with 4 GPU(s), using 4 stream(s) per device with 4 group(s) of 16 CPU worker thread(s)
[PB Info 2026-Mar-02 18:55:16] # 40  0  0  0  0   0 pool:  0 0 bases/GPU/minute: 0.0 
[PB Info 2026-Mar-02 18:55:26] # 40  0  0  0  0   0 pool:  0 0 bases/GPU/minute: 0.0 
[PB Info 2026-Mar-02 18:55:36] #  0 31  0 39  0   7 pool: 28 2005969467 bases/GPU/minute: 300895420

Please visit https://docs.nvidia.com/clara/#parabricks for detailed documentation




The beginning of this script defines paths to the files needed to run fq2bam. For this example we will provide a reference `.fasta`, known sites `.vcf`, and the two reads as `.fastq` files. We will also needs output paths for the resulting `.bam` containing the aligned reads and the `recal.txt` with the recalibration scores.  

At the end of of this script is the run command. Every command in Parabricks starts with `pbrun` followed by the name of the tool to run and then any arguments. Most of the arguments are file inputs as defined above, however there are two boolean flags at the end that will improve the performance of the tool. 

The coordinate sorting step of alignment can be moved to the GPU by including `--gpusort`. The GPU can also help writing the output `.bam` file by enabling the `--gpuwrite` flag. 

In this example, we are running with a minimal set of arguments but there are dozens of extra options outlined in the [documentation](https://docs.nvidia.com/clara/parabricks/latest/documentation/tooldocs/man_fq2bam.html#fq2bam-reference). 

Once the cell executing fq2bam has finished running, we can look inside the `data/output` folder and see the that `.bam` file containing our aligned reads has been generated. 

In [8]:
%%sh 

ls data/output 

NIST7035_TAAGGCGA_L001_R1_001.bam
NIST7035_TAAGGCGA_L001_R1_001.bam.bai
NIST7035_TAAGGCGA_L001_R1_001_chrs.txt
recal.txt


Now let's use samtools view to inspect the `.bam` file. 

In [ ]:
%%sh

samtools view -H data/output/NIST7035_TAAGGCGA_L001_R1_001.bam | tail 
echo "---"
samtools view data/output/NIST7035_TAAGGCGA_L001_R1_001.bam | head -

@SQ	SN:HLA-DRB1*15:01:01:02	LN:11571
@SQ	SN:HLA-DRB1*15:01:01:03	LN:11056
@SQ	SN:HLA-DRB1*15:01:01:04	LN:11056
@SQ	SN:HLA-DRB1*15:02:01	LN:10313
@SQ	SN:HLA-DRB1*15:03:01:01	LN:11567
@SQ	SN:HLA-DRB1*15:03:01:02	LN:11569
@SQ	SN:HLA-DRB1*16:02:01	LN:11005
@RG	ID:H7AP8ADXX.1	LB:lib1	PL:bar	SM:sample	PU:H7AP8ADXX.1
@PG	ID:pbrun fq2bam	PN:pbrun fq2bam	VN:4.6.0-1	CL:pbrun fq2bam --ref ref/Homo_sapiens_assembly38.fasta --in-fq NIST7035_TAAGGCGA_L001_R1_001.fastq.gz NIST7035_TAAGGCGA_L001_R2_001.fastq.gz --knownSites ref/Homo_sapiens_assembly38.known_indels.vcf.gz --out-bam output/NIST7035_TAAGGCGA_L001_R1_001.bam --out-recal-file output/recal.txt --gpusort --gpuwrite --tmp-dir /workdir
@PG	ID:samtools	PN:samtools	PP:pbrun fq2bam	VN:1.19.2	CL:samtools view -H data/output/NIST7035_TAAGGCGA_L001_R1_001.bam
---
HWI-D00119:50:H7AP8ADXX:1:2208:5166:2843	99	chr1	10002	0	101M	=	10013	112	AACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCC	;@?BDADFFDAHG

The output is split into two sections separated by `---`.

### Header (`samtools view -H`)

The header contains lines starting with `@` tags:

| Tag | Meaning |
|-----|---------|
| `@HD` | File-level metadata — `VN` is the SAM format version, `SO` is sort order (e.g. `coordinate`) |
| `@SQ` | One line per reference sequence — `SN` is the chromosome name, `LN` is its length in bases |
| `@RG` | Read group — `ID`, `SM` (sample), `PL` (platform), `LB` (library), `PU` (platform unit) |
| `@PG` | Program record — records which tools processed the file and with what arguments |

### Alignment records (`samtools view | head -5`)

Each read is one tab-delimited line with 11 mandatory fields:

```
QNAME  FLAG  RNAME  POS  MAPQ  CIGAR  RNEXT  PNEXT  TLEN  SEQ  QUAL
```

| Field | Description |
|-------|-------------|
| `QNAME` | Read name |
| `FLAG` | Bitwise flag encoding read properties (paired, mapped, strand, etc.) |
| `RNAME` | Reference chromosome the read mapped to (e.g. `chr1`) |
| `POS` | 1-based leftmost mapping position |
| `MAPQ` | Mapping quality (255 = not available; higher = more confident) |
| `CIGAR` | Alignment description — `M`=match, `I`=insertion, `D`=deletion, `S`=soft clip |
| `RNEXT` | Chromosome of the mate read (`=` means same as `RNAME`) |
| `PNEXT` | Position of the mate read |
| `TLEN` | Observed template length (insert size; negative if read is downstream) |
| `SEQ` | Base sequence |
| `QUAL` | Phred-scaled base quality scores (ASCII-encoded) |

### Decoding the FLAG

The `FLAG` field is a sum of bit values. Common ones:

| Value | Meaning |
|-------|---------|
| `1` | Read is paired |
| `2` | Both reads mapped in proper pair |
| `4` | Read is unmapped |
| `16` | Read is on the reverse strand |
| `1024` | Read is a PCR/optical duplicate |
| `2048` | Supplementary alignment (chimeric read) |

For example, `FLAG=99` = 1+2+32+64, meaning: paired, proper pair, mate on reverse strand, read 1.

### CIGAR string example

`101M` → 101 bases all matched/mismatched to the reference (typical for a clean 101 bp read).  
`50M2I49M` → 50 bases matched, 2-base insertion relative to reference, 49 more bases matched.

Now that the samples are aligned, we can detect the variants. 


## Run variant calling

In this section, we will use the [Parabricks DeepVariant](https://docs.nvidia.com/clara/parabricks/latest/documentation/tooldocs/man_deepvariant.html) tool to call variants on the `.bam` we generated using fq2bam.  The code to do this is contained in the bash script located at `scripts/deepvariant.sh`. The following cell executes this code. It will take around 10 minutes to run, so kick it off now and read below about what the script is doing. 

In [12]:
%%sh 

docker run --rm --gpus all \
    --volume $(pwd)/data:/workdir \
    --workdir /workdir \
    nvcr.io/nvidia/clara/clara-parabricks:4.6.0-1 \
    pbrun deepvariant \
    --ref ref/Homo_sapiens_assembly38.fasta \
    --in-bam output/NIST7035_TAAGGCGA_L001_R1_001.bam \
    --out-variants output/NIST7035_TAAGGCGA_L001_R1_001.vcf \
    --use-wes-model 

[Parabricks Options Mesg]: Setting --num-streams-per-gpu based on available device memory.
[Parabricks Options Mesg]: Please consider using --run-partition for best performance with more than 2 GPUs
Detected 4 CUDA Capable device(s), considering 4 device(s)
  CUDA Driver Version / Runtime Version          13.0 / 12.9
Using model for CUDA Capability Major/Minor version number:    80


[PB Info 2026-Mar-02 19:11:46] ------------------------------------------------------------------------------
[PB Info 2026-Mar-02 19:11:46] ||                 Parabricks accelerated Genomics Pipeline                 ||
[PB Info 2026-Mar-02 19:11:46] ||                              Version 4.6.0-1                             ||
[PB Info 2026-Mar-02 19:11:46] ||                                deepvariant                               ||
[PB Info 2026-Mar-02 19:11:46] ------------------------------------------------------------------------------
[PB Info 2026-Mar-02 19:11:46] Starting DeepVariant
[PB Info 2026-Mar-02 19:11:46] Running with 4 GPU devices, each with 2 group instances and 6 workers
[PB Info 2026-Mar-02 19:11:46] ProgressMeter -	Current-Locus	Elapsed-Minutes
[PB Info 2026-Mar-02 19:11:52] ProgressMeter -	chr1:14000	0.1
[PB Info 2026-Mar-02 19:11:58] ProgressMeter -	chr1:216287000	0.2
[PB Info 2026-Mar-02 19:12:04] ProgressMeter -	chr3:178030000	0.3
[PB Info 2026-Mar-02 19:12

/usr/local/parabricks/binaries/bin/deepsomatic 4 2 --ref /workdir/ref/Homo_sapiens_assembly38.fasta --reads /workdir/output/NIST7035_TAAGGCGA_L001_R1_001.bam -o /workdir/output/NIST7035_TAAGGCGA_L001_R1_001.vcf -n 6 --model /usr/local/parabricks/binaries/model/80+/shortread/deepvariant_wes.eng --channel_insert_size --pileup_image_width 221 --max_reads_per_partition 1500 --partition_size 1000 --vsc_min_count_snps 2 --vsc_min_count_indels 2 --vsc_min_fraction_snps 0.12 --min_mapping_quality 5 --min_base_quality 10 --alt_aligned_pileup none --variant_caller VERY_SENSITIVE_CALLER --dbg_min_base_quality 15 --ws_min_windows_distance 80 --aux_fields_to_keep HP --p_error 0.001 --max_ins_size 10
Deepvariant done, total time: 1.1 min
Please visit https://docs.nvidia.com/clara/#parabricks for detailed documentation



DeepVariant is a variant caller that uses a convolutional neural network (CNN) as opposed to bayesian statistics to discover variants. This method often performs better, especially on low frequency variants. It has the added benefit that it can be retrained to improve performance on any dataset. In this blueprint, we will use the [GPU optimized version of DeepVariant found in Parabricks](https://docs.nvidia.com/clara/parabricks/latest/documentation/tooldocs/man_deepvariant.html).  

Just like the fq2bam script, the beginning of the script defines the file paths needed to run Parabricks DeepVariant. This time we only need the reference `.fasta`, the aligned reads `.bam` file generated by fq2bam in the previous section, and the path to the output `.vcf` to store the variants. 

At the end of the script is the run command. Just as before, it starts with `pbrun` but this time instead of `fq2bam` we are running `deepvariant` with a minimal set of arguments. The file inputs are familiar, but let's look into the additional flag at the end. 

DeepVariant is based on a CNN model. Different CNNs can be trained on different datasets. By default, Parabricks DeepVariant uses a model trained on whole genome (WGS) data, however in this blueprint we are using an exome and must therefore use a CNN trained on whole exome (WES) data. To do this, we use the `--use-wes-model` argument. 

For more information on all the flags, check out the [documentation](https://docs.nvidia.com/clara/parabricks/latest/documentation/tooldocs/man_deepvariant.html#deepvariant-reference). 

The output of DeepVariant is a Variant Call Format (VCF) file which represents all the variants found in the sample in a tabular format. The column names are standardized as follows: 

| Column Name | Value |
|---|---|
| CHROM | Reference sequence name (chromosome or contig) |
| POS | 1-based leftmost position of the variant on CHROM |
| ID | Variant identifier(s) (e.g., rsID) or `.` if none |
| REF | Reference allele sequence at POS |
| ALT | Comma-separated alternate allele(s) observed at POS |
| QUAL | Phred-scaled quality score for the assertion of the ALT allele(s) |
| FILTER | `PASS` if variant passed filters, or semicolon-separated failing filter names |
| INFO | Semicolon-separated annotations (flags or key=value pairs) with additional site-level metadata |
| FORMAT | Colon-separated keys defining the per-sample fields (e.g., `GT:DP:AD`) |
| sample | Sample-specific values corresponding to FORMAT (e.g., genotype, depth, allele depths)

This file also contains a header with meta data about the samples. For more information on the VCF file format, review the [spec](https://samtools.github.io/hts-specs/VCFv4.2.pdf). 

[bcftools](https://github.com/samtools/bcftools) is a powerful command line utility to view and manipulate vcf files. Use [bcftools view](https://samtools.github.io/bcftools/bcftools.html#view) to inspect the contents of our VCF file. 

In [13]:
%%sh 

bcftools view --no-header data/output/NIST7035_TAAGGCGA_L001_R1_001.vcf | head

chr1	14210	.	G	A	22.7	PASS	.	GT:GQ:DP:AD:VAF:PL	1/1:21:3:1,2:0.666667:22,24,0
chr1	14397	.	CTGT	C	12.6	PASS	.	GT:GQ:DP:AD:VAF:PL	0/1:12:30:20,10:0.333333:12,0,21
chr1	14574	.	A	G	1.8	RefCall	.	GT:GQ:DP:AD:VAF:PL	./.:5:3:0,3:1:0,13,3
chr1	14590	.	G	A	0.2	RefCall	.	GT:GQ:DP:AD:VAF:PL	./.:13:3:0,3:1:0,19,13
chr1	14599	.	T	A	0	RefCall	.	GT:GQ:DP:AD:VAF:PL	0/0:21:3:0,3:1:0,24,24
chr1	14604	.	A	G	0	RefCall	.	GT:GQ:DP:AD:VAF:PL	0/0:21:3:0,3:1:0,24,24
chr1	14610	.	T	C	0.2	RefCall	.	GT:GQ:DP:AD:VAF:PL	./.:13:4:0,3:0.75:0,19,14
chr1	14653	.	C	T	0.1	RefCall	.	GT:GQ:DP:AD:VAF:PL	./.:19:23:6,17:0.73913:0,24,20
chr1	14677	.	G	A	0.1	RefCall	.	GT:GQ:DP:AD:VAF:PL	./.:18:34:20,14:0.411765:0,19,23
chr1	15118	.	A	G	1.6	RefCall	.	GT:GQ:DP:AD:VAF:PL	./.:5:15:4,10:0.666667:0,15,3


The entire end-to-end germline workflow has now been run using Parabricks in this notebook environment, from `.fastq` to `.vcf`. 

## Next Steps


This notebook serves as an introduction for how to use Parabricks. There are many other resources for more in-depth tutorials such as [Cloud Usage Guides](https://docs.nvidia.com/clara/parabricks/latest/tutorials/cloudguides.html) for getting Parabricks up and running on popular clouds such as AWS, GCP, and Azure. There is also a [GitHub repository](https://github.com/clara-parabricks-workflows) full of Parabricks workflows examples such as [WDL](https://github.com/clara-parabricks-workflows/parabricks-wdl) and [NextFlow](https://github.com/clara-parabricks-workflows/parabricks-nextflow) workflows and even a [notebook](https://github.com/clara-parabricks-workflows/deepvariant-retraining-notebook/blob/main/Retraining_DeepVariant.ipynb) showing how to train customer DeepVariant models. 